In [ ]:
!pip install torch torchvision torchaudio

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

In [ ]:
training_data = [
    ("Alice lives in Paris".split(), ["PER","O","O","LOC"]),
    ("Bob works at Microsoft".split(), ["PER","O","O","ORG"]),
    ("Ravi studies at IIT Mumbai".split(), ["PER","O","O","ORG","LOC"]),
    ("Google is based in California".split(), ["ORG","O","O","O","LOC"]),
    ("Emma visited New York".split(), ["PER","O","LOC","LOC"]),
    ("Amazon hires engineers".split(), ["ORG","O","O"]),
    ("Rahul works in Infosys".split(), ["PER","O","O","ORG"]),
    ("Tesla is an innovative company".split(), ["ORG","O","O","O","O"]),
    ("Priya lives in Delhi".split(), ["PER","O","O","LOC"]),
    ("Facebook is a social media company".split(), ["ORG","O","O","O","O","O"])
]

In [ ]:
word_to_ix = {}
tag_to_ix = {"PER":0, "O":1, "LOC":2, "ORG":3}

for sentence, tags in training_data:
    for word in sentence:
        if word not in word_to_ix:
            word_to_ix[word] = len(word_to_ix)
print(word_to_ix)

{'Alice': 0, 'lives': 1, 'in': 2, 'Paris': 3, 'Bob': 4, 'works': 5, 'at': 6, 'Microsoft': 7, 'Ravi': 8, 'studies': 9, 'IIT': 10, 'Mumbai': 11, 'Google': 12, 'is': 13, 'based': 14, 'California': 15, 'Emma': 16, 'visited': 17, 'New': 18, 'York': 19, 'Amazon': 20, 'hires': 21, 'engineers': 22, 'Rahul': 23, 'Infosys': 24, 'Tesla': 25, 'an': 26, 'innovative': 27, 'company': 28, 'Priya': 29, 'Delhi': 30, 'Facebook': 31, 'a': 32, 'social': 33, 'media': 34}


In [ ]:
def prepare_sequence(seq, to_ix):
    idxs = [to_ix[w] for w in seq]
    return torch.tensor(idxs, dtype=torch.long)

In [ ]:
class LSTMTagger(nn.Module):

    def __init__(self, embedding_dim, hidden_dim, vocab_size, tagset_size):
        super(LSTMTagger, self).__init__()

        self.embedding = nn.Embedding(vocab_size, embedding_dim)

        self.lstm = nn.LSTM(embedding_dim, hidden_dim)

        self.hidden2tag = nn.Linear(hidden_dim, tagset_size)

    def forward(self, sentence):

        embeds = self.embedding(sentence)

        lstm_out, _ = self.lstm(embeds.view(len(sentence), 1, -1))

        tag_space = self.hidden2tag(lstm_out.view(len(sentence), -1))

        tag_scores = torch.log_softmax(tag_space, dim=1)

        return tag_scores

In [ ]:
EMBEDDING_DIM = 6
HIDDEN_DIM = 6

model = LSTMTagger(EMBEDDING_DIM, HIDDEN_DIM, len(word_to_ix), len(tag_to_ix))
loss_function = nn.NLLLoss()
optimizer = optim.SGD(model.parameters(), lr=0.1)

In [ ]:
for epoch in range(100):

    for sentence, tags in training_data:

        model.zero_grad()

        sentence_in = prepare_sequence(sentence, word_to_ix)
        targets = prepare_sequence(tags, tag_to_ix)

        tag_scores = model(sentence_in)

        loss = loss_function(tag_scores, targets)

        loss.backward()
        optimizer.step()

In [ ]:
with torch.no_grad():
    inputs = prepare_sequence("Alice works at Google".split(), word_to_ix)
    tag_scores = model(inputs)
    print(tag_scores)

tensor([[-0.0907, -4.0113, -4.7221, -2.8179],
        [-5.2915, -0.0823, -5.7752, -2.6467],
        [-6.6398, -0.0801, -5.1003, -2.6651],
        [-6.7383, -0.4413, -4.6669, -1.0607]])
